In [ ]:
%pip install requests

In [1]:
import requests
import socket

In [2]:
#Make the call directly

url = "http://flowdemo-endpoint-bhdubngdh9gfd8e6.z01.azurefd.net"

# Get the User-Agent from requests default headers
user_agent = requests.utils.default_headers()['User-Agent']

# Make the request
response = requests.get(url)

# Get local IP address
hostname = socket.gethostname()
ip_address = socket.gethostbyname(hostname)

print(f"Status Code: {response.status_code}")
print(f"URL opened: {url}")
print(f"IP Address: {ip_address}")
print(f"User-Agent: {user_agent}")

Status Code: 200
URL opened: http://flowdemo-endpoint-bhdubngdh9gfd8e6.z01.azurefd.net
IP Address: 192.168.86.45
User-Agent: python-requests/2.34.2


In [3]:
# Squid already running and listening on that port

import requests
import socket


def make_reuqest_on_port(port):
    # Set up proxy (squid running on localhost:port)
    proxies = {
        'http': f'http://localhost:{port}',
        'https': f'http://localhost:{port}',
    }

    url = "http://flowdemo-endpoint-bhdubngdh9gfd8e6.z01.azurefd.net"

    # Get the User-Agent from requests default headers
    user_agent = requests.utils.default_headers()['User-Agent']

    # Make the request through the proxy
    response = requests.get(url, proxies=proxies, timeout=20)

    # Get local IP address
    hostname = socket.gethostname()
    ip_address = socket.gethostbyname(hostname)

    print(f"Status Code: {response.status_code}")
    print(f"URL opened: {url}")
    print(f"IP Address (local): {ip_address}")
    print(f"Proxy Used: localhost:{port}")
    print(f"User-Agent: {user_agent}")


In [4]:
#dynamically restarting squid for each port and makes a request

from pathlib import Path
import re
import shutil
import subprocess
import time

def restart_squid_with_new_port(new_port):
    candidates = [
        Path('/opt/homebrew/etc/squid.conf'),
        Path('/usr/local/etc/squid.conf'),
        Path('/etc/squid/squid.conf'),
    ]

    config_path = next((p for p in candidates if p.exists()), None)
    if not config_path:
        raise FileNotFoundError('Could not find squid.conf in common locations.')

    # Backup before editing
    backup_path = config_path.with_suffix(config_path.suffix + '.bak')
    shutil.copy2(config_path, backup_path)

    config_text = config_path.read_text()
    if re.search(r'^\s*http_port\s+\S+', config_text, flags=re.MULTILINE):
        updated_text = re.sub(
            r'^\s*http_port\s+\S+',
            f'http_port {new_port}',
            config_text,
            count=1,
            flags=re.MULTILINE,
        )
    else:
        updated_text = f'http_port {new_port}\n' + config_text

    config_path.write_text(updated_text)
    subprocess.run(['brew', 'services', 'restart', 'squid'], check=True)

    # Squid can take a moment to bind after restart; retry before failing.
    last_stdout = ''
    last_stderr = ''
    for _ in range(12):
        check = subprocess.run(
            ['lsof', '-nP', f'-iTCP:{new_port}', '-sTCP:LISTEN'],
            capture_output=True,
            text=True,
            check=False,
        )
        last_stdout = check.stdout
        last_stderr = check.stderr
        if check.returncode == 0 and check.stdout.strip():
            print(f'Updated {config_path} to use http_port {new_port}')
            print(f'Backup created at {backup_path}')
            print('Squid restarted successfully.')
            print(check.stdout.strip())
            break
        time.sleep(0.5)
    else:
        raise RuntimeError(
            f'Squid was restarted but nothing is listening on port {new_port}.\n{last_stdout}{last_stderr}'
        )
    


In [5]:
new_port = 3130
restart_squid_with_new_port(new_port)
make_reuqest_on_port(new_port)

Stopping `squid`... (might take a while)
==> Successfully stopped `squid` (label: homebrew.mxcl.squid)
==> Successfully started `squid` (label: homebrew.mxcl.squid)
Updated /opt/homebrew/etc/squid.conf to use http_port 3130
Backup created at /opt/homebrew/etc/squid.conf.bak
Squid restarted successfully.
COMMAND   PID       USER   FD   TYPE             DEVICE SIZE/OFF NODE NAME
squid   66549 jamiedixon   15u  IPv6 0x2a5d5d9b58c499b4      0t0  TCP *:3130 (LISTEN)
Status Code: 200
URL opened: http://flowdemo-endpoint-bhdubngdh9gfd8e6.z01.azurefd.net
IP Address (local): 192.168.86.45
Proxy Used: localhost:3130
User-Agent: python-requests/2.34.2


In [6]:
import time

for port in range(3140, 3190): #5000 is already in use
    print(f"starting {port}")
    restart_squid_with_new_port(port)
    make_reuqest_on_port(port)
    time.sleep(5)

starting 3140
Stopping `squid`... (might take a while)
==> Successfully stopped `squid` (label: homebrew.mxcl.squid)
==> Successfully started `squid` (label: homebrew.mxcl.squid)
Updated /opt/homebrew/etc/squid.conf to use http_port 3140
Backup created at /opt/homebrew/etc/squid.conf.bak
Squid restarted successfully.
COMMAND   PID       USER   FD   TYPE             DEVICE SIZE/OFF NODE NAME
squid   66829 jamiedixon   15u  IPv6 0xe1a8a9c8729f208f      0t0  TCP *:3140 (LISTEN)
Status Code: 200
URL opened: http://flowdemo-endpoint-bhdubngdh9gfd8e6.z01.azurefd.net
IP Address (local): 192.168.86.45
Proxy Used: localhost:3140
User-Agent: python-requests/2.34.2
starting 3141
Stopping `squid`... (might take a while)
==> Successfully stopped `squid` (label: homebrew.mxcl.squid)
==> Successfully started `squid` (label: homebrew.mxcl.squid)
Updated /opt/homebrew/etc/squid.conf to use http_port 3141
Backup created at /opt/homebrew/etc/squid.conf.bak
Squid restarted successfully.
COMMAND   PID     